# Spatial SVG benchmark on real data (DLPFC) — Colab runner

Branch `spatial-svg-benchmark` of `wpuzhu/BIOINF-590-Individual-Project`.

This notebook:
1. installs the pipeline (and, optionally, GPU domain methods: GraphST, STAGATE, SpaGCN);
2. downloads human DLPFC Visium slices (Maynard et al. 2021; 12 slices with manual layers) from Hugging Face, with SODB as fallback;
3. runs the benchmark: domain aggregation (R1), double-dipping null (R2), spike-in (R3), adjacent-slice replication (R4), perturbations (R5) and global SVG references (R6);
4. writes a self-contained `report.html` and zips all results.

**Before running:** *Runtime → Change runtime type → T4 GPU* (only needed for `gpu_quick` / `full`).

| Profile | Slices | Methods | Approx. time (T4) |
|---|---|---|---|
| `quick` | 151673 + 151674 | 5 CPU methods | ~30–45 min |
| `gpu_quick` | 151673 + 151674 | CPU + GraphST, STAGATE, SpaGCN | ~1–1.5 h |
| `full` | all 12 (6 adjacent pairs) | CPU + GPU, B = 20 | ~8–12 h (run with Drive so it can resume) |

Results are checkpointed per slice. If Colab disconnects, re-run the notebook: finished slices are skipped.

In [ ]:
# ---- settings ----
PROFILE = "quick"          # "quick" | "gpu_quick" | "full"
USE_DRIVE = False          # True: save results to Google Drive (recommended for "full")
SAMPLES = None             # e.g. "151673,151674" to override the profile's slices
BRANCH = "spatial-svg-benchmark"
REPO = "https://github.com/wpuzhu/BIOINF-590-Individual-Project.git"

In [ ]:
import os, subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "No GPU visible (fine for PROFILE='quick').")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_DIR = f"/content/drive/MyDrive/svg_benchmark/{PROFILE}"
else:
    RESULTS_DIR = f"/content/results/{PROFILE}"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("results ->", RESULTS_DIR)

In [ ]:
%cd /content
!rm -rf BIOINF-590-Individual-Project
!git clone -q --depth 1 -b $BRANCH $REPO
%cd /content/BIOINF-590-Individual-Project/spatial_benchmark
!git log --oneline -1

In [ ]:
# core dependencies (CPU pipeline + data download)
!pip install -q anndata scanpy huggingface_hub scikit-misc

In [ ]:
# optional GPU domain methods (skipped for PROFILE == "quick")
if PROFILE != "quick":
    import torch
    tv = torch.__version__.split("+")[0]
    cu = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
    print("torch", tv, cu)
    # GraphST (POT for optimal transport utilities); --no-deps avoids its old version pins
    !pip install -q POT
    !pip install -q --no-deps git+https://github.com/JinmiaoChenLab/GraphST.git
    # SpaGCN (we use its k-means initialisation, so louvain is not needed)
    !pip install -q --no-deps SpaGCN
    # STAGATE (PyG version): needs torch_geometric + torch_sparse wheels matching torch/CUDA
    !pip install -q torch_geometric
    !pip install -q torch_sparse -f https://data.pyg.org/whl/torch-{tv}+{cu}.html
    !pip install -q --no-deps git+https://github.com/QIFEIDKN/STAGATE_pyG.git
    import sys; sys.path.insert(0, ".")
    from stbench.real import gpu_methods as G
    ok, problems = G.available()
    print("GPU methods available:", list(ok))
    print("problems (these methods will be skipped):", problems)

In [ ]:
# sanity checks: unit tests + a 1-minute simulated run of the full pipeline
!python -m unittest discover -s tests 2>&1 | tail -3
!python scripts/run_real.py --config configs/sim_test.json --out /content/sim_check 2>&1 | grep -v -i warning | tail -4

In [ ]:
# data check: load one slice and confirm counts, coordinates and layer labels were found
import sys; sys.path.insert(0, ".")
from stbench.real import io
adata, src = io.load_dlpfc("151673", cache_dir="data")
print("source:", src, "| shape:", adata.shape)
print("obs columns:", list(adata.obs.columns)[:25])
print("label column detected:", io.find_label(adata))
ctx = io.to_context(adata)
print({k: ctx.extra[k] for k in ["count_source", "label_col", "n_spots", "n_genes", "label_names"]})
# If the label column is wrong, set "label_col" in configs/real_<profile>.json.
# If both download sources fail, upload DLPFC_151673.h5ad etc. to /content/BIOINF-590-Individual-Project/spatial_benchmark/data/

In [ ]:
# ---- the real-data run ----
extra = f"--samples {SAMPLES}" if SAMPLES else ""
!python scripts/run_real.py --config configs/real_{PROFILE}.json --out "$RESULTS_DIR" $extra 2>&1 | grep -v -i warning

In [ ]:
from IPython.display import HTML, display
report = os.path.join(RESULTS_DIR, "report.html")
display(HTML(open(report).read()))

In [ ]:
# download everything (report + per-slice CSVs + gene lists)
import shutil
zip_path = shutil.make_archive(f"/content/svg_benchmark_{PROFILE}", "zip", RESULTS_DIR)
from google.colab import files
files.download(zip_path)

## What to send back / what to look at

* `report.html`: all results with automatic interpretation sentences.
* `<slice>/domain_svgs.csv`: per gene, selection probability `pi`, whether SplitStab selected it, its modal domain, and whether the Kang et al. pipeline called it.
* `<slice>/global_svgs.csv`: global SVG calls for each reference (intersection, ACAT, Castl-like) and the per-test p-values.
* `r4_replication.csv`: adjacent-section reproducibility.

Upload the zip (or push it to the branch under `spatial_benchmark/results/`) and the analysis can be continued from there.